In [179]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import  pandas as pd
import numpy as np
import pickle

Se importan las librerias

In [180]:
df = pd.read_csv("../data/air_quality_processed.csv")
df = df.drop(columns=["Name","Start_Date"])
scaler=StandardScaler()
columnas=["Geo Join ID","Data Value","year","month","day_of_week"]
df[columnas] = scaler.fit_transform(df[columnas])

print(df)

       Geo Join ID  Data Value      year     month  day_of_week  target
0        -0.076478   -0.520820  1.316632 -0.066889    -0.963537      12
1        -0.076464   -0.059178  1.526271  1.270801     0.046199      13
2        -0.076453    0.340966  1.316632 -0.066889    -0.963537      16
3        -0.076453   -0.105632  1.316632  1.270801    -0.458669      13
4        -0.076466   -0.232198  0.897353 -0.066889     1.055935      13
...            ...         ...       ...       ...          ...     ...
18857    -0.076491   -0.014920 -2.037599 -1.181632     1.055935      17
18858    -0.076465   -0.248324 -2.037599 -1.181632     1.055935      17
18859    -0.076477   -0.715132 -2.037599 -1.181632     1.055935      15
18860    -0.076490   -0.787275 -2.037599 -1.181632     1.055935      15
18861    -0.076466    1.831312 -2.037599 -1.181632     1.055935       0

[18862 rows x 6 columns]


Se carga el dataset con la data preprocesada

In [181]:
labeled, unlabeled = train_test_split(df, test_size=0.80)

Dividimos el dataset

In [182]:
X_unlabeled = unlabeled.drop(columns=["target"])
y_unlabeled = unlabeled["target"]
X_labeled = labeled.drop(columns=["target"])
y_labeled = labeled["target"]

In [183]:
with open("../data/X_labeled.pkl", "wb") as f:
    pickle.dump(X_labeled, f)
with open("../data/y_labeled.pkl", "wb") as f:
    pickle.dump(y_labeled, f)
with open("../data/X_unlabeled.pkl", "wb") as f:
    pickle.dump(X_unlabeled, f)
with open("../data/y_unlabeled.pkl", "wb") as f:
    pickle.dump(y_unlabeled, f)

Ahora hacemos la implementacion de nuestro modelo Random Forest y usamos algunas metricas para evaluar su desempeño con el calculo de la media entre diferentes Semillas

In [184]:
seeds = [10, 42, 100, 500, 999]
scores = []

results = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': []
}

for s in seeds:
    model = RandomForestClassifier(n_estimators=100, random_state=s)

    model.fit(X_labeled, y_labeled)

    y_pred = model.predict(X_unlabeled)
    y_probs = model.predict_proba(X_unlabeled)[:, 1]

    results['accuracy'].append(accuracy_score(y_unlabeled, y_pred))
    results['precision'].append(precision_score(y_unlabeled, y_pred, average='macro'))
    results['recall'].append(recall_score(y_unlabeled, y_pred, average='macro'))
    results['f1'].append(f1_score(y_unlabeled, y_pred, average='macro'))

print(f"Resultados finales tras {len(seeds)} ejecuciones:")
for metric, values in results.items():
    mean = np.mean(values)
    std = np.std(values)
    print(f"{metric.capitalize()}: {mean:.4f} ± {std:.4f}")

Resultados finales tras 5 ejecuciones:
Accuracy: 0.8342 ± 0.0005
Precision: 0.5249 ± 0.0029
Recall: 0.5085 ± 0.0019
F1: 0.5114 ± 0.0021


Despues con Logistic Regression

In [185]:
scores = []

results2 = {
    'accuracy': [],
    'precision': [],
    'recall': [],
    'f1': []
}

for s in seeds:
    modelII = LogisticRegression(max_iter=1000,random_state=s)

    modelII.fit(X_labeled, y_labeled)

    y_pred = modelII.predict(X_unlabeled)
    y_probs = modelII.predict_proba(X_unlabeled)[:, 1]

    results2['accuracy'].append(accuracy_score(y_unlabeled, y_pred))
    results2['precision'].append(precision_score(y_unlabeled, y_pred, average='macro',zero_division=0))
    results2['recall'].append(recall_score(y_unlabeled, y_pred, average='macro'))
    results2['f1'].append(f1_score(y_unlabeled, y_pred, average='macro'))

print(f"Resultados finales tras {len(seeds)} ejecuciones:")
for metric, values in results2.items():
    mean = np.mean(values)
    std = np.std(values)
    print(f"{metric.capitalize()}: {mean:.4f} ± {std:.4f}")

Resultados finales tras 5 ejecuciones:
Accuracy: 0.7487 ± 0.0000
Precision: 0.2315 ± 0.0000
Recall: 0.2651 ± 0.0000
F1: 0.2380 ± 0.0000
